### Запуск Spark

In [1]:
from dags import strava_kafka
from numpy.distutils.conv_template import header
from pyspark import RDD
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import *
import pandas as pd
import datetime
import os


spark = SparkSession \
            .builder \
            .config("spark.master", "local") \
            .config('spark.executor.cores', '8')\
            .config('spark.executor.memory', '2g')\
            .config('spark.driver.memory', '2g')\
            .config('spark.dynamicAllocation.minExecutors', '4')\
            .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
            .config("spark.jars.packages", "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2") \
            .appName("spark_halltape") \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
hadoop_conf.set("fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


print("Активные Spark сессии:", 'http://localhost:4040/jobs')

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-19603aba-507b-4fbc-8b45-84baf8cdfbe4;1.0
	confs: [default]
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found commons-codec#commons-codec;1.11 in central
	found org.apache.httpcomponents#httpmime;4.5.13 in central
:: resolution report :: resolve 355ms :: artifacts dl 15ms
	:: modules in use:
	com.clickhouse#clickhouse-client;0.3.2 from central in [default]
	

Активные Spark сессии: http://localhost:4040/jobs


In [2]:
# Создадим dataframe словарь
# Путь к файлу в S3
s3_path = "s3a://prod/api/earthquake/*.json"
s3_path_regions = "s3a://prod/jdbc/regions/*.parquet"

# Чтение JSON-файла
df = spark.read.json(s3_path)
regions = spark.read.parquet(s3_path_regions)

# # Покажем несколько строк
# df.show(truncate=False)


flattened = df.selectExpr("explode(features) as feature") \
    .select(
        col("feature.id").alias("id"),
        from_unixtime(col("feature.properties.time") / 1000).alias("time"),
        col("time").cast("date").alias("load_date"),
        trim(split(col("feature.properties.place"), ",").getItem(0)).alias("place"),
        trim(split(col("feature.properties.place"), ",").getItem(1)).alias("initial_region"), 
        col("feature.properties.mag").alias("magnitude"),
        col("feature.properties.felt").alias("felt"),
        col("feature.properties.tsunami").alias("tsunami"),
        col("feature.properties.url").alias("url"),
        col("feature.geometry.coordinates")[0].alias("longitude"),
        col("feature.geometry.coordinates")[1].alias("latitude"),
        col("feature.geometry.coordinates")[2].alias("depth"),
        md5(lower(trim(col("feature.properties.place")))).alias("place_hash")
    )


flattened.alias("f")\
    .join(regions.alias("r"), "place_hash", "left")\
    .where("initial_region IS NULL") \
    .drop('initial_region', 'place_hash') \
    .show(truncate=False)

# felt - Сколько людей сообщили, что ощутили землетрясение (в данном случае 1 человек)
# tsunami - Флаг цунами: 1 — возможно, 0 — нет
# url - Ссылка на страницу события на сайте USGS


25/05/17 21:14:58 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://prod/api/earthquake/*.json.
java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:53)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRe

Py4JJavaError: An error occurred while calling o40.json.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(Da

In [ ]:
ch_max_df = spark.read \
    .format("jdbc") \
    .option("url", 'jdbc:clickhouse://clickhouse:8123/default',) \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("dbtable", f"(SELECT MAX(load_date) as max_date FROM enriched_earthquakes)") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .load()

max_date = ch_max_df.collect()[0]["max_date"].strftime("%Y-%m-%d")

if max_date == '1970-01-01':
    predicate = "1=1"
else:
    predicate = f"load_date > '{max_date}'"

predicate

In [ ]:
spark.stop()

### Сборка витрины данных

In [ ]:
PATH = '/home/jovyan/source'

def debit_card(date):
    card = spark.read.csv(f"{PATH}/card.csv", header=True, sep=";")\
                .where(f''' load_date = "{date}" ''')
    
    status_card = spark.read.csv(f"{PATH}/Status_card.csv", header=True, sep=";")\
                        .where(f''' load_date = "{date}"  ''')
    
    transactions = spark.read.csv(f"{PATH}/transactions.csv", header=True, sep=";")\
                        .where(f''' load_date = "{date}"  ''')
    
    
    first_trx = transactions.groupBy('card_num')\
                            .agg(F.min('transaction_datetime').alias('transaction_datetime'))
    
    
    first_trx_info = first_trx.join(transactions, ['card_num','transaction_datetime'], 'inner')
    
    dt_trx = first_trx_info.select('card_num', 'amount', '*')
    df_st = status_card.select('card_num', 'card_num_md5', 'status')
    
    result_df = dt_trx\
                    .join(df_st, "card_num", "inner")\
                    .join(card, "card_num_md5", "right")\
                    .drop('card_num_md5')
    
    final_df = result_df\
                    .where(''' card_num IS NOT NULL ''')\
                    .groupBy('card_num',
                             'transaction_datetime',
                             'status',
                             'card_order_dt',
                             'url',
                             'cookie')\
                    .agg(F.max('amount').alias('amt'))
    
    
    datamart = final_df\
                    .select('card_order_dt',
                            'card_num',
                            'cookie',
                            'url',
                            'amt',
                            'status')\
                    .withColumn('transaction_level',
                                    F.when(F.col('amt') > 300, True).otherwise(False))\
                    .withColumn('status_flag',
                                    F.when(F.col('status') == "выдана", True).otherwise(False))\
                    .withColumn('partition_date', F.lit(date).cast('string').alias('partition_date'))\
                    .withColumn('load_date', F.lit(date).cast('string').alias('load_date'))\
                    .drop('amt', 'status')
    
    datamart.write.mode("append").partitionBy('partition_date').csv('data_lake/debit_card', header=True)



start = '2024-07-01'
end = '2024-07-10'

for dt in pd.date_range(start, end):
    date = str(dt)[:10]
    print(date)
    debit_card(date)

PYSPARK

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("n_bainin") \
    .config("spark.executor.cores", "8") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print(spark.sparkContext.uiWebUrl)

http://1a4957a6466b:4040


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35356)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/opt/conda/lib/python3.11/site-packages/pyspark/accumulators.py", line 281, in handle
    poll(accum_updates)
  File "/opt/conda/lib/python3.11/site-packages/pyspark/accumulators.py", line 253, in poll
    if func():
       ^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pyspark/accumulators.py", line 257, in accum_updates
    num_u

In [39]:
PATH = 'data_lake/data/customs_data.csv'

In [41]:
df = spark.read.csv(PATH, sep=';', header=True)

# df.show(truncate=False)

df.show(2, False, True)

# df.count()

-RECORD 0--------------------------------------
 month         | 01/2016                       
 country       | IT                            
 code          | 6204695000                    
 value         | 131                           
 netto         | 1                             
 quantity      | 7                             
 region        | 46000                         
 district      | 01                            
 direction_eng | IM                            
 measure_eng   | ShT                           
 load_date     | 2024-07-01T00:00:00.000+03:00 
-RECORD 1--------------------------------------
 month         | 01/2016                       
 country       | CN                            
 code          | 9001900009                    
 value         | 112750                        
 netto         | 18                            
 quantity      | 0                             
 region        | 46000                         
 district      | 01                     

In [42]:
df.printSchema()

root
 |-- month: string (nullable = true)
 |-- country: string (nullable = true)
 |-- code: string (nullable = true)
 |-- value: string (nullable = true)
 |-- netto: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- direction_eng: string (nullable = true)
 |-- measure_eng: string (nullable = true)
 |-- load_date: string (nullable = true)



In [43]:
result = df \
    .withColumnRenamed("direction_eng", "direction") \
    .withColumnRenamed("measure_eng", "measure")

# result.printSchema()
result.columns

['month',
 'country',
 'code',
 'value',
 'netto',
 'quantity',
 'region',
 'district',
 'direction',
 'measure',
 'load_date']

In [46]:
df_country_distnct = result \
    .select("country") \
    .distinct()

df_country_distnct.rdd.getNumPartitions()

1

In [57]:
# result.show(2, False, True)

df_country_distnct.printSchema()

# res = df_country_distnct.groupBy().agg(
#     F.count("country").alias("cnt_countries")
# )
#
# res.show()

root
 |-- country: string (nullable = true)



+-------------+
|cnt_countries|
+-------------+
|          253|
+-------------+



In [38]:
result \
    .groupBy('country') \
    .agg(F.count('*').alias('total_rows')) \
    .orderBy(F.col('total_rows').desc()) \
    .show( truncate=False)

NameError: name 'result' is not defined

In [8]:
result.cache()

result.show()

26/05/27 20:11:37 WARN CacheManager: Asked to cache already cached data.


+-------+-------+----------+------+------+--------+------+--------+---------+-------+--------------------+
|  month|country|      code| value| netto|quantity|region|district|direction|measure|           load_date|
+-------+-------+----------+------+------+--------+------+--------+---------+-------+--------------------+
|01/2016|     IT|6204695000|   131|     1|       7| 46000|      01|       IM|    ShT|2024-07-01T00:00:...|
|01/2016|     CN|9001900009|112750|    18|       0| 46000|      01|       IM|      1|2024-01-01T00:00:...|
|01/2016|     BY|8414302004|   392|    57|       8| 50000|      06|       IM|    ShT|2024-06-01T00:00:...|
|01/2016|     US|9018509000| 54349|   179|       0| 40000|      02|       IM|      1|2024-04-01T00:00:...|
|01/2016|     EE|9021101000| 17304|   372|       0| 46000|      01|       IM|      1|2024-02-01T00:00:...|
|01/2016|     FR|3816000000|323488|253600|       0| 40000|      02|       IM|      1|2024-02-01T00:00:...|
|01/2016|     MX|8523519300|  1611|  

In [37]:
df_de = result \
    .where(F.col('country') == 'DE') \
    .where(F.col('value').isNotNull())

df_de.show()

NameError: name 'result' is not defined

In [79]:
df_de2 = result \
    .where(''' country == "DE" ''') \
    .where(''' value is not null ''')

df.show()

+-------+-------+----------+------+------+--------+------+--------+-------------+-----------+--------------------+
|  month|country|      code| value| netto|quantity|region|district|direction_eng|measure_eng|           load_date|
+-------+-------+----------+------+------+--------+------+--------+-------------+-----------+--------------------+
|01/2016|     IT|6204695000|   131|     1|       7| 46000|      01|           IM|        ShT|2024-07-01T00:00:...|
|01/2016|     CN|9001900009|112750|    18|       0| 46000|      01|           IM|          1|2024-01-01T00:00:...|
|01/2016|     BY|8414302004|   392|    57|       8| 50000|      06|           IM|        ShT|2024-06-01T00:00:...|
|01/2016|     US|9018509000| 54349|   179|       0| 40000|      02|           IM|          1|2024-04-01T00:00:...|
|01/2016|     EE|9021101000| 17304|   372|       0| 46000|      01|           IM|          1|2024-02-01T00:00:...|
|01/2016|     FR|3816000000|323488|253600|       0| 40000|      02|           IM

In [13]:
df_de = result \
    .where(F.col('country') == 'DE') \
    .where(F.col('value').isNotNull())

df_de2 = result \
    .where(''' country == "DE" ''') \
    .where(''' value is not null ''')

print(df_de.count() == df_de2.count())

True


In [14]:
result.columns

['month',
 'country',
 'code',
 'value',
 'netto',
 'quantity',
 'region',
 'district',
 'direction',
 'measure',
 'load_date']

In [17]:
res = result\
    .select('month',
             'country',
             'code',
             'value',
             'netto',
             'quantity',
             'region',
             'district',
             'direction',
             'measure',
             'load_date',
            F.col('load_date').cast('date').alias('load_dt'))

res.show(2, False, True)

-RECORD 0----------------------------------
 month     | 01/2016                       
 country   | IT                            
 code      | 6204695000                    
 value     | 131                           
 netto     | 1                             
 quantity  | 7                             
 region    | 46000                         
 district  | 01                            
 direction | IM                            
 measure   | ShT                           
 load_date | 2024-07-01T00:00:00.000+03:00 
 load_dt   | 2024-07-01                    
-RECORD 1----------------------------------
 month     | 01/2016                       
 country   | CN                            
 code      | 9001900009                    
 value     | 112750                        
 netto     | 18                            
 quantity  | 0                             
 region    | 46000                         
 district  | 01                            
 direction | IM                 

In [20]:
# res\
#     .write\
#     .format('csv')\
#     .options(header=True, sep=';')\
#     .csv('synthetic_data/data')

partitions_num = res.rdd.getNumPartitions()
print(f"Количество партиций: {partitions_num}")

Количество партиций: 16


In [24]:
res\
    .coalesce(1)\
    .write\
    .format('csv')\
    .options(header=True, sep=';')\
    .csv('synthetic_data/data_coalesce')

In [30]:
res\
    .write\
    .partitionBy('load_dt')\
    .format('csv')\
    .options(header=True, sep=';')\
    .csv('synthetic_data/data_partitioned')

In [34]:
res\
    .repartition(1, 'load_date')\
    .write\
    .partitionBy('load_dt')\
    .format('csv')\
    .options(header=True, sep=';')\
    .csv('synthetic_data/data_repartitioned')

In [39]:
reader_no_control = spark\
    .read\
    .csv('synthetic_data/data', header=True, sep=';')\
    .where(''' load_dt == "2024-01-01" ''')

reader_no_control.count()

5998326

In [41]:
reader_final_one_file = spark\
    .read\
    .csv('synthetic_data/data_coalesce', header=True, sep=';')\
    .where(''' load_dt == "2024-01-01" ''')

reader_final_one_file.count()

5998326

In [52]:
reader_partitioned = spark\
    .read\
    .csv('synthetic_data/data_partitioned', header=True, sep=';')\
    .where(''' load_dt == "2024-01-01" ''')

reader_partitioned.cache()

DataFrame[month: string, country: string, code: string, value: string, netto: string, quantity: string, region: string, district: string, direction: string, measure: string, load_date: string, load_dt: date]

In [46]:
reader_partitioned_repart = spark\
    .read\
    .csv('synthetic_data/data_repartitioned', header=True, sep=';')\
    .where(''' load_dt == "2024-01-01" ''')

reader_partitioned_repart.count()

5998326

In [3]:
data = [
    (14000, "Северный"),
    (11000, "Южный"),
    (10000, "Восточный"),
    (26000, "Западный"),
    (56000, "Центральный"),
]

region_df = spark.createDataFrame(data, schema='region_id long, name string')

region_df.show()

customs_data = spark\
    .read\
    .csv("data_lake/data/customs_data.csv", header=True, sep=';')

customs_data.where(''' region ==  26000 ''').show(10, False, True)

+---------+-----------+
|region_id|       name|
+---------+-----------+
|    14000|   Северный|
|    11000|      Южный|
|    10000|  Восточный|
|    26000|   Западный|
|    56000|Центральный|
+---------+-----------+

-RECORD 0--------------------------------------
 month         | 01/2016                       
 country       | CN                            
 code          | 7326903000                    
 value         | 661                           
 netto         | 265                           
 quantity      | 0                             
 region        | 26000                         
 district      | 08                            
 direction_eng | IM                            
 measure_eng   | 1                             
 load_date     | 2024-01-01T00:00:00.000+03:00 
-RECORD 1--------------------------------------
 month         | 01/2016                       
 country       | CN                            
 code          | 6702100000                    
 value         

In [4]:
import time

start_time = time.time()

customs_data\
    .join(
    F.broadcast(region_df),
    customs_data.region==region_df.region_id,
    "left"
).count()

end_time = time.time()

print(f"{end_time - start_time:.2f}")

6.92


In [5]:
customs_data.cache().show()

+-------+-------+----------+------+------+--------+------+--------+-------------+-----------+--------------------+
|  month|country|      code| value| netto|quantity|region|district|direction_eng|measure_eng|           load_date|
+-------+-------+----------+------+------+--------+------+--------+-------------+-----------+--------------------+
|01/2016|     IT|6204695000|   131|     1|       7| 46000|      01|           IM|        ShT|2024-07-01T00:00:...|
|01/2016|     CN|9001900009|112750|    18|       0| 46000|      01|           IM|          1|2024-01-01T00:00:...|
|01/2016|     BY|8414302004|   392|    57|       8| 50000|      06|           IM|        ShT|2024-06-01T00:00:...|
|01/2016|     US|9018509000| 54349|   179|       0| 40000|      02|           IM|          1|2024-04-01T00:00:...|
|01/2016|     EE|9021101000| 17304|   372|       0| 46000|      01|           IM|          1|2024-02-01T00:00:...|
|01/2016|     FR|3816000000|323488|253600|       0| 40000|      02|           IM

In [49]:
df_filtered = df\
    .where(""" month != '01/2016' """)\
    .cache().show()

26/05/27 21:20:06 WARN CacheManager: Asked to cache already cached data.


+-------+-------+----------+------+------+--------+------+--------+-------------+-----------+--------------------+
|  month|country|      code| value| netto|quantity|region|district|direction_eng|measure_eng|           load_date|
+-------+-------+----------+------+------+--------+------+--------+-------------+-----------+--------------------+
|02/2016|     AT|5510110000|  2590|   469|       0| 24000|      01|           IM|          1|2024-01-01T00:00:...|
|02/2016|     US|3209100009|   353|   109|       0| 46000|      01|           IM|          1|2024-06-01T00:00:...|
|02/2016|     PT|6110309900|    74|     2|       5| 58000|      02|           IM|        ShT|2024-02-01T00:00:...|
|02/2016|     ES|3004900002| 68081|  1011|       0| 46000|      01|           IM|          1|2024-02-01T00:00:...|
|02/2016|     RO|6402999100|  2221|    71|     137| 45000|      01|           IM|        PAR|2024-02-01T00:00:...|
|02/2016|     HU|8481807399|  1484|     4|       0| 46000|      01|           IM

In [39]:
df.show(1)

+-------+-------+----------+-----+-----+--------+------+--------+-------------+-----------+--------------------+
|  month|country|      code|value|netto|quantity|region|district|direction_eng|measure_eng|           load_date|
+-------+-------+----------+-----+-----+--------+------+--------+-------------+-----------+--------------------+
|01/2016|     IT|6204695000|  131|    1|       7| 46000|      01|           IM|        ShT|2024-07-01T00:00:...|
+-------+-------+----------+-----+-----+--------+------+--------+-------------+-----------+--------------------+
only showing top 1 row



In [10]:
customs_data.unpersist()

DataFrame[month: string, country: string, code: string, value: string, netto: string, quantity: string, region: string, district: string, direction_eng: string, measure_eng: string, load_date: string]

In [7]:
from pyspark.storagelevel import StorageLevel

customs_data.persist(StorageLevel.DISK_ONLY).count()

26392290

In [11]:
customs_data.cache().count()

26392290

In [15]:
data = [
    (1,'one'),
    (2,'two'),
    (3,'three'),
    (4,'four'),
    (5,'five'),
    (6,'six'),
    (7, 'seven'),
    (8, 'eight'),
    (9, 'nine'),
]

df = spark.createDataFrame(data, schema=['id', 'number'])

df.show()

+---+------+
| id|number|
+---+------+
|  1|   one|
|  2|   two|
|  3| three|
|  4|  four|
|  5|  five|
|  6|   six|
|  7| seven|
|  8| eight|
|  9|  nine|
+---+------+



In [28]:
# df.rdd.getNumPartitions()
df.rdd.glom().collect()

[[],
 [Row(id=1, number='one')],
 [Row(id=2, number='two')],
 [Row(id=3, number='three')],
 [],
 [Row(id=4, number='four')],
 [Row(id=5, number='five')],
 [Row(id=6, number='six')],
 [],
 [Row(id=7, number='seven')],
 [Row(id=8, number='eight')],
 [Row(id=9, number='nine')]]

In [30]:
mix = df.repartition(8)
mix.rdd.getNumPartitions()
mix.rdd.glom().collect()

[[Row(id=2, number='two'), Row(id=4, number='four')],
 [],
 [Row(id=1, number='one'), Row(id=8, number='eight')],
 [],
 [Row(id=6, number='six')],
 [],
 [Row(id=3, number='three'),
  Row(id=7, number='seven'),
  Row(id=9, number='nine')],
 [Row(id=5, number='five')]]

In [34]:
mix.repartition(6).rdd.glom().collect()

[[Row(id=3, number='three'), Row(id=8, number='eight')],
 [],
 [Row(id=4, number='four')],
 [Row(id=5, number='five'),
  Row(id=7, number='seven'),
  Row(id=9, number='nine')],
 [Row(id=1, number='one')],
 [Row(id=2, number='two'), Row(id=6, number='six')]]

In [36]:
mix.coalesce(3).rdd.glom().collect()

[[Row(id=2, number='two'), Row(id=4, number='four')],
 [Row(id=1, number='one'),
  Row(id=8, number='eight'),
  Row(id=3, number='three'),
  Row(id=7, number='seven'),
  Row(id=9, number='nine')],
 [Row(id=6, number='six'), Row(id=5, number='five')]]

In [48]:
mix.toPandas().head()

,id,number
0,2,two
1,4,four
2,1,one
3,8,eight
4,6,six


In [52]:
mix\
    .filter(F.col('number') == 1)\
    .repartition(1)

DataFrame[id: bigint, number: string]

kafka strava

In [3]:
from pyspark.sql import SparkSession
minio_endpoint = 'http://localhost:9002'

spark = SparkSession.builder \
    .appName("n_bainin") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 13:43:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
